# Model Training - Medical Multimodal Retrieval

This notebook demonstrates the training process for the multimodal retrieval model.

In [ ]:
import os
import sys
sys.path.append('../app')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import our modules
from models.multimodal_model import MultimodalModel
from data.data_loader import ChestXrayDataset, get_train_transforms, get_val_transforms
from utils.seed import set_seed

# Set random seed for reproducibility
set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
# Training configuration
config = {
    'batch_size': 16,
    'learning_rate': 1e-5,
    'epochs': 20,
    'image_size': 224,
    'max_length': 128,
    'embed_dim': 512,
    'weight_decay': 1e-4,
    'temperature': 0.07
}

print("Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## Load Data

In [ ]:
# Paths
BASE_DIR = r"C:\Users\sagar\OneDrive\Desktop\IISC\data\mimic_cxr_project"
TRAIN_CSV = os.path.join(BASE_DIR, "processed", "train_processed.csv")
VAL_CSV = os.path.join(BASE_DIR, "processed", "val_processed.csv")

# Load datasets
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print(f"Train samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")

# Create datasets
train_dataset = ChestXrayDataset(
    train_df,
    transforms=get_train_transforms(config['image_size'])
)

val_dataset = ChestXrayDataset(
    val_df,
    transforms=get_val_transforms(config['image_size'])
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## Initialize Model

In [ ]:
# Initialize model
model = MultimodalModel().to(device)

# Freeze encoders, unfreeze last 2 layers
for param in model.image_encoder.parameters():
    param.requires_grad = False
for param in model.text_encoder.parameters():
    param.requires_grad = False

# Unfreeze last 2 layers
for param in model.image_encoder.vision_model.encoder.layers[-2:].parameters():
    param.requires_grad = True
for param in model.text_encoder.encoder.layer[-2:].parameters():
    param.requires_grad = True

# Keep projection heads trainable
for param in model.image_projection.parameters():
    param.requires_grad = True
for param in model.text_projection.parameters():
    param.requires_grad = True

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable ratio: {trainable_params/total_params*100:.2f}%")

## Training Setup

In [ ]:
# Loss function
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, image_embeddings, text_embeddings):
        logits = (image_embeddings @ text_embeddings.T) / self.temperature
        labels = torch.arange(logits.shape[0]).to(device)
        
        loss_i = F.cross_entropy(logits, labels)
        loss_t = F.cross_entropy(logits.T, labels)
        
        return (loss_i + loss_t) / 2

criterion = ContrastiveLoss(temperature=config['temperature'])

# Optimizer
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config['epochs']
)

print("Training components initialized!")

## Training Loop

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(config['epochs']):
    print(f"\nEpoch {epoch+1}/{config['epochs']}")
    print("-" * 50)
    
    # Training
    model.train()
    train_loss = 0.0
    
    for batch in tqdm(train_loader, desc="Training"):
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        optimizer.zero_grad()
        
        image_embeddings, text_embeddings = model(images, input_ids, attention_mask)
        loss = criterion(image_embeddings, text_embeddings)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            image_embeddings, text_embeddings = model(images, input_ids, attention_mask)
            loss = criterion(image_embeddings, text_embeddings)
            
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    scheduler.step()
    
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), '../checkpoints/best_model.pth')
        print("✅ Best model saved!")
    
print("\nTraining completed!")

## Training Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(np.array(val_losses) - np.array(train_losses), label='Val-Train Gap', linewidth=2, color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss Difference')
plt.title('Overfitting Monitor')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final training loss: {train_losses[-1]:.4f}")
print(f"Final validation loss: {val_losses[-1]:.4f}")

## Summary

This notebook demonstrates:
- Model initialization with frozen encoders
- Contrastive loss training
- Training and validation loops
- Learning rate scheduling
- Model checkpointing
- Training progress visualization

The trained model can now be used for retrieval tasks.